In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay, classification_report, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# 1. Load Data & Define Target
df = pd.read_csv("donor_features_v2.csv")
# Fill NaN in interval metric with a sentinel value (-1 indicates no repeat gifts yet)
df["avg_days_between_donations"] = df["avg_days_between_donations"].fillna(-1)

# Fill any remaining NaNs in numeric features with 0
df = df.fillna(0)

df["is_active_90d"] = (df["donations_last_90_days"] > 0).astype(int)

# 2. Select Features (Excludes direct 90-day leakage features)
feature_cols = [
    # Base Metrics
    "donation_count",
    "total_donations",
    "average_donation",
    "campaign_count",
    # Historical Cadence & Tenure
    "days_since_first_donation",
    "donor_tenure_days",
    "avg_days_between_donations",
    # Annual Rolling Metrics
    "donations_last_365_days",
    "revenue_last_365_days",
    # Velocity & Acceleration Indicators
    "frequency_velocity_365d",
    "monetary_velocity_365d",
]

X = df[feature_cols]
y = df["is_active_90d"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Define Candidate Models
models = {
    "Logistic Regression": Pipeline(
        [("scaler", StandardScaler()), ("clf", LogisticRegression(C=1.0))]
    ),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost (Baseline)": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        eval_metric="logloss",
        random_state=42,
    ),
}

# 4. Hyperparameter Tuning for XGBoost
xgb_param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.8, 1.0],
}

grid_xgb = GridSearchCV(
    XGBClassifier(eval_metric="logloss", random_state=42),
    xgb_param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
)
grid_xgb.fit(X_train, y_train)

models["XGBoost (Tuned)"] = grid_xgb.best_estimator_

# 5. Train & Evaluate Models
plt.figure(figsize=(9, 6))
results = []

for name, model in models.items():
    if name != "XGBoost (Tuned)":
        model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)

    results.append(
        {
            "Model": name,
            "ROC-AUC": round(auc, 4),
        }
    )

    # Plot ROC Curve
    RocCurveDisplay.from_predictions(
        y_test, y_proba, name=f"{name} (AUC = {auc:.4f})", ax=plt.gca()
    )

plt.plot([0, 1], [0, 1], "k--", label="Chance Level")
plt.title(
    "ROC-AUC: Expanded Feature Set Benchmarking",
    fontsize=13,
    fontweight="bold",
)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# 6. Display Performance Summary Table
df_comparison = pd.DataFrame(results).sort_values(
    by="ROC-AUC", ascending=False
)
print("=== Model Performance Comparison ===")
print(df_comparison.to_string(index=False))

# 7. XGBoost Feature Importance Breakdown
tuned_xgb = models["XGBoost (Tuned)"]
xgb_importances = pd.Series(
    tuned_xgb.feature_importances_, index=X.columns
).sort_values(ascending=False)

print("\n=== XGBoost Feature Importances ===")
print(xgb_importances.round(4).to_string())